# COVID-QU-Ex — Offline Augmentation & Private Dataset Export

Augments **all splits** (Train / Val / Test) of both tasks:
- **Lung Segmentation** — lung masks
- **Infection Segmentation** — infection masks

Uses **10 augmentation variants per source image** — enough diversity without
generating millions of files.

Output is written **directly into zip files** (no loose PNGs on disk),
then published as a **private Kaggle dataset** via the API.

**Setup** — Add these two Secrets in the notebook sidebar (Add-ons → Secrets):
- `KAGGLE_USERNAME` → your Kaggle username
- `KAGGLE_KEY`      → your Kaggle API key

In [ ]:

import os, shutil, glob

working_dir   = '/kaggle/working'
deleted_files = deleted_dirs = 0
for item in glob.glob(os.path.join(working_dir, '*')):
    if os.path.isfile(item) or os.path.islink(item):
        os.remove(item);      deleted_files += 1
    elif os.path.isdir(item):
        shutil.rmtree(item);  deleted_dirs  += 1

print(f'Cleared /kaggle/working/')
print(f'  Deleted files : {deleted_files}')
print(f'  Deleted dirs  : {deleted_dirs}')
print(f'  Free space    : {shutil.disk_usage(working_dir).free / 1e9:.1f} GB')

In [ ]:

!pip install albumentations -q

import os, glob, random, time, zipfile, json, io, shutil, math
import cv2
import numpy as np
from PIL import Image
import albumentations as A

print(f'Albumentations : {A.__version__}')
print(f'Free space     : {shutil.disk_usage("/kaggle/working").free / 1e9:.1f} GB')

In [ ]:

LUNG_ROOT    = '/kaggle/input/datasets/anasmohammedtahir/covidqu/Lung Segmentation Data/Lung Segmentation Data'
INFECT_ROOT  = '/kaggle/input/datasets/anasmohammedtahir/covidqu/Infection Segmentation Data/Infection Segmentation Data'
WORK_DIR     = '/kaggle/working'
ZIP_DIR      = os.path.join(WORK_DIR, 'zips')
DATASET_DIR  = os.path.join(WORK_DIR, 'dataset_to_publish')

IMG_SIZE          = 256
SEED              = 42
SPLITS            = ['Train', 'Val', 'Test']
CATEGORIES        = ['COVID-19', 'Non-COVID', 'Normal']
VARIANTS_PER_PAIR = 5  
                           
                             
                           
PAIRS_PER_CHUNK   = 8000   
DATASET_SLUG  = 'covidqu-augmented'
DATASET_TITLE = 'COVID-QU-Ex Augmented Segmentation'

random.seed(SEED)
np.random.seed(SEED)
os.makedirs(ZIP_DIR,     exist_ok=True)
os.makedirs(DATASET_DIR, exist_ok=True)

print(f'Variants per pair : {VARIANTS_PER_PAIR}')
print(f'Pairs per chunk   : {PAIRS_PER_CHUNK}')
print(f'Splits            : {SPLITS}')
print(f'Categories        : {CATEGORIES}')

In [ ]:

def collect_pairs(root, mask_folder_name):
    pairs = []
    for split in SPLITS:
        for cat in CATEGORIES:
            img_dir  = os.path.join(root, split, cat, 'images')
            mask_dir = os.path.join(root, split, cat, mask_folder_name)
            if not os.path.isdir(img_dir):
                print(f'  [SKIP] {split}/{cat}/images not found')
                continue
            for img_path in sorted(glob.glob(os.path.join(img_dir, '*.png'))):
                fname     = os.path.basename(img_path)
                mask_path = os.path.join(mask_dir, fname)
                if os.path.exists(mask_path):
                    pairs.append((img_path, mask_path, split, cat))
    return pairs


lung_pairs   = collect_pairs(LUNG_ROOT,   'lung masks')
infect_pairs = collect_pairs(INFECT_ROOT, 'infection masks')

print(f'Lung source pairs      : {len(lung_pairs):,}')
print(f'Infection source pairs : {len(infect_pairs):,}')
print(f'\nEstimated output:')
print(f'  Lung      : {len(lung_pairs) * VARIANTS_PER_PAIR:,} images')
print(f'  Infection : {len(infect_pairs) * VARIANTS_PER_PAIR:,} images')
print(f'  Total     : {(len(lung_pairs) + len(infect_pairs)) * VARIANTS_PER_PAIR:,} images')
print(f'\nLung breakdown:')
for split in SPLITS:
    for cat in CATEGORIES:
        n = sum(1 for p in lung_pairs if p[2]==split and p[3]==cat)
        if n: print(f'  {split:<6} / {cat:<12} : {n:,}')
print(f'\nInfection breakdown:')
for split in SPLITS:
    for cat in CATEGORIES:
        n = sum(1 for p in infect_pairs if p[2]==split and p[3]==cat)
        if n: print(f'  {split:<6} / {cat:<12} : {n:,}')

In [ ]:
def enhance_xray(image: np.ndarray) -> np.ndarray:
    gray  = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY) if image.ndim == 3 else image
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    eq    = clahe.apply(gray)
    blur  = cv2.GaussianBlur(eq, (0, 0), sigmaX=2)
    sharp = cv2.addWeighted(eq, 1.5, blur, -0.5, 0)
    return cv2.cvtColor(sharp, cv2.COLOR_GRAY2RGB)


def build_pipelines(img_size):
    _R = A.Resize(img_size, img_size)
    BC = cv2.BORDER_CONSTANT

    # v2.x normalized helpers
    def GN(lo, hi):  return A.GaussNoise(std_range=(lo/255, hi/255), p=1.0)
    def CD(h, n, w): return A.CoarseDropout(num_holes_range=(1,h), hole_height_range=(8,n), hole_width_range=(8,w), fill=0, p=1.0)
    def DS(lo, hi):  return A.Downscale(scale_range=(lo, hi), p=1.0)
    def SOL(t):      return A.Solarize(threshold_range=(t/255, t/255), p=1.0)

    return {
        'orig':               A.Compose([_R]),
        'hflip':              A.Compose([_R, A.HorizontalFlip(p=1.0)]),
        'rot_15':             A.Compose([_R, A.Rotate(limit=(15,15),    border_mode=BC, p=1.0)]),
        'rot_neg15':          A.Compose([_R, A.Rotate(limit=(-15,-15),  border_mode=BC, p=1.0)]),
        'rot_30':             A.Compose([_R, A.Rotate(limit=(30,30),    border_mode=BC, p=1.0)]),
        'rot_neg30':          A.Compose([_R, A.Rotate(limit=(-30,-30),  border_mode=BC, p=1.0)]),
        'rot_5':              A.Compose([_R, A.Rotate(limit=(5,5),      border_mode=BC, p=1.0)]),
        'rot_neg5':           A.Compose([_R, A.Rotate(limit=(-5,-5),    border_mode=BC, p=1.0)]),
        'rot_45':             A.Compose([_R, A.Rotate(limit=(45,45),    border_mode=BC, p=1.0)]),
        'rot_neg45':          A.Compose([_R, A.Rotate(limit=(-45,-45),  border_mode=BC, p=1.0)]),
        'elastic_soft':       A.Compose([_R, A.ElasticTransform(alpha=40,  sigma=5,  p=1.0)]),
        'elastic_hard':       A.Compose([_R, A.ElasticTransform(alpha=120, sigma=12, p=1.0)]),
        'elastic_mid':        A.Compose([_R, A.ElasticTransform(alpha=80,  sigma=8,  p=1.0)]),
        'grid_distort':       A.Compose([_R, A.GridDistortion(num_steps=5, distort_limit=0.3, p=1.0)]),
        'grid_distort_hard':  A.Compose([_R, A.GridDistortion(num_steps=5, distort_limit=0.5, p=1.0)]),
        'optical_distort':    A.Compose([_R, A.OpticalDistortion(distort_limit=0.3, p=1.0)]),
        'optical_distort_hard':A.Compose([_R, A.OpticalDistortion(distort_limit=0.5, p=1.0)]),
        'bright_up':          A.Compose([_R, A.RandomBrightnessContrast(brightness_limit=(0.3,0.3),   contrast_limit=0,           p=1.0)]),
        'bright_down':        A.Compose([_R, A.RandomBrightnessContrast(brightness_limit=(-0.3,-0.3), contrast_limit=0,           p=1.0)]),
        'bright_up2':         A.Compose([_R, A.RandomBrightnessContrast(brightness_limit=(0.5,0.5),   contrast_limit=0,           p=1.0)]),
        'bright_down2':       A.Compose([_R, A.RandomBrightnessContrast(brightness_limit=(-0.5,-0.5), contrast_limit=0,           p=1.0)]),
        'contrast_up':        A.Compose([_R, A.RandomBrightnessContrast(brightness_limit=0, contrast_limit=(0.3,0.3),   p=1.0)]),
        'contrast_down':      A.Compose([_R, A.RandomBrightnessContrast(brightness_limit=0, contrast_limit=(-0.3,-0.3), p=1.0)]),
        'gamma_up':           A.Compose([_R, A.RandomGamma(gamma_limit=(120,150), p=1.0)]),
        'gamma_down':         A.Compose([_R, A.RandomGamma(gamma_limit=(60,80),   p=1.0)]),
        'gamma_mid':          A.Compose([_R, A.RandomGamma(gamma_limit=(90,110),  p=1.0)]),
        'clahe':              A.Compose([_R, A.CLAHE(clip_limit=4.0, p=1.0)]),
        'clahe2':             A.Compose([_R, A.CLAHE(clip_limit=8.0, p=1.0)]),
        'equalize':           A.Compose([_R, A.Equalize(p=1.0)]),
        'solarize':           A.Compose([_R, SOL(128)]),
        'solarize2':          A.Compose([_R, SOL(64)]),
        'gauss_blur':         A.Compose([_R, A.GaussianBlur(blur_limit=(5,9),   p=1.0)]),
        'gauss_blur2':        A.Compose([_R, A.GaussianBlur(blur_limit=(11,15), p=1.0)]),
        'motion_blur':        A.Compose([_R, A.MotionBlur(blur_limit=9,  p=1.0)]),
        'motion_blur2':       A.Compose([_R, A.MotionBlur(blur_limit=15, p=1.0)]),
        'median_blur':        A.Compose([_R, A.MedianBlur(blur_limit=5,  p=1.0)]),
        'median_blur2':       A.Compose([_R, A.MedianBlur(blur_limit=9,  p=1.0)]),
        'gauss_noise':        A.Compose([_R, GN(20, 80)]),
        'gauss_noise2':       A.Compose([_R, GN(80, 160)]),
        'iso_noise':          A.Compose([_R, A.ISONoise(color_shift=(0.01,0.05), intensity=(0.1,0.4), p=1.0)]),
        'iso_noise2':         A.Compose([_R, A.ISONoise(color_shift=(0.05,0.10), intensity=(0.4,0.8), p=1.0)]),
        'sharpen':            A.Compose([_R, A.Sharpen(alpha=(0.3,0.6), lightness=(0.8,1.0), p=1.0)]),
        'sharpen2':           A.Compose([_R, A.Sharpen(alpha=(0.6,1.0), lightness=(0.8,1.0), p=1.0)]),
        'coarse_dropout':     A.Compose([_R, CD(8,  24, 24)]),
        'coarse_dropout2':    A.Compose([_R, CD(16, 32, 32)]),
        'grid_dropout':       A.Compose([_R, A.GridDropout(ratio=0.3, p=1.0)]),
        'grid_dropout2':      A.Compose([_R, A.GridDropout(ratio=0.5, p=1.0)]),
        'downscale':          A.Compose([_R, DS(0.5,  0.75)]),
        'downscale2':         A.Compose([_R, DS(0.25, 0.5)]),
        'affine':             A.Compose([_R, A.Affine(scale=(0.85,1.15), translate_percent=(-0.1,0.1),  shear=(-10,10), p=1.0)]),
        'affine2':            A.Compose([_R, A.Affine(scale=(0.70,1.30), translate_percent=(-0.15,0.15),shear=(-15,15), p=1.0)]),

        'hflip_rot15':        A.Compose([_R, A.HorizontalFlip(p=1.0), A.Rotate(limit=(15,15),   border_mode=BC, p=1.0)]),
        'hflip_rot_neg15':    A.Compose([_R, A.HorizontalFlip(p=1.0), A.Rotate(limit=(-15,-15), border_mode=BC, p=1.0)]),
        'hflip_rot30':        A.Compose([_R, A.HorizontalFlip(p=1.0), A.Rotate(limit=(30,30),   border_mode=BC, p=1.0)]),
        'hflip_rot_neg30':    A.Compose([_R, A.HorizontalFlip(p=1.0), A.Rotate(limit=(-30,-30), border_mode=BC, p=1.0)]),
        'hflip_elastic':      A.Compose([_R, A.HorizontalFlip(p=1.0), A.ElasticTransform(alpha=60,  sigma=6,  p=1.0)]),
        'hflip_elastic_hard': A.Compose([_R, A.HorizontalFlip(p=1.0), A.ElasticTransform(alpha=120, sigma=12, p=1.0)]),
        'hflip_grid':         A.Compose([_R, A.HorizontalFlip(p=1.0), A.GridDistortion(num_steps=5, distort_limit=0.3, p=1.0)]),
        'hflip_affine':       A.Compose([_R, A.HorizontalFlip(p=1.0), A.Affine(scale=(0.85,1.15), translate_percent=(-0.1,0.1), shear=(-10,10), p=1.0)]),
        'hflip_optical':      A.Compose([_R, A.HorizontalFlip(p=1.0), A.OpticalDistortion(distort_limit=0.3, p=1.0)]),
        'hflip_downscale':    A.Compose([_R, A.HorizontalFlip(p=1.0), DS(0.5, 0.75)]),
        
        'elastic_bright':     A.Compose([_R, A.ElasticTransform(alpha=60, sigma=6, p=1.0), A.RandomBrightnessContrast(0.2, 0.2, p=1.0)]),
        'elastic_gamma':      A.Compose([_R, A.ElasticTransform(alpha=60, sigma=6, p=1.0), A.RandomGamma(gamma_limit=(80,140), p=1.0)]),
        'elastic_clahe':      A.Compose([_R, A.ElasticTransform(alpha=60, sigma=6, p=1.0), A.CLAHE(clip_limit=4.0, p=1.0)]),
        'elastic_noise':      A.Compose([_R, A.ElasticTransform(alpha=60, sigma=6, p=1.0), GN(20, 80)]),
        'elastic_blur':       A.Compose([_R, A.ElasticTransform(alpha=60, sigma=6, p=1.0), A.GaussianBlur(blur_limit=(3,7), p=1.0)]),
        'elastic_sharpen':    A.Compose([_R, A.ElasticTransform(alpha=60, sigma=6, p=1.0), A.Sharpen(alpha=(0.3,0.6), lightness=(0.8,1.0), p=1.0)]),
        'elastic_motion':     A.Compose([_R, A.ElasticTransform(alpha=60, sigma=6, p=1.0), A.MotionBlur(blur_limit=9, p=1.0)]),
        'elastic_solarize':   A.Compose([_R, A.ElasticTransform(alpha=60, sigma=6, p=1.0), SOL(128)]),
        'elastic_equalize':   A.Compose([_R, A.ElasticTransform(alpha=60, sigma=6, p=1.0), A.Equalize(p=1.0)]),
        'elastic_dropout':    A.Compose([_R, A.ElasticTransform(alpha=60, sigma=6, p=1.0), CD(8, 24, 24)]),
        
        'rot_bright':         A.Compose([_R, A.Rotate(limit=12, border_mode=BC, p=1.0), A.RandomBrightnessContrast(0.2, 0.2, p=1.0)]),
        'rot_noise':          A.Compose([_R, A.Rotate(limit=12, border_mode=BC, p=1.0), GN(10, 40)]),
        'rot_clahe':          A.Compose([_R, A.Rotate(limit=12, border_mode=BC, p=1.0), A.CLAHE(clip_limit=4.0, p=1.0)]),
        'rot_gamma':          A.Compose([_R, A.Rotate(limit=12, border_mode=BC, p=1.0), A.RandomGamma(gamma_limit=(80,140), p=1.0)]),
        'rot_blur':           A.Compose([_R, A.Rotate(limit=12, border_mode=BC, p=1.0), A.GaussianBlur(blur_limit=(3,7), p=1.0)]),
        'rot_motion':         A.Compose([_R, A.Rotate(limit=12, border_mode=BC, p=1.0), A.MotionBlur(blur_limit=9, p=1.0)]),
        'rot_sharpen':        A.Compose([_R, A.Rotate(limit=12, border_mode=BC, p=1.0), A.Sharpen(alpha=(0.3,0.6), lightness=(0.8,1.0), p=1.0)]),
        'rot_equalize':       A.Compose([_R, A.Rotate(limit=12, border_mode=BC, p=1.0), A.Equalize(p=1.0)]),
        'rot_solarize':       A.Compose([_R, A.Rotate(limit=12, border_mode=BC, p=1.0), SOL(128)]),
        'rot_dropout':        A.Compose([_R, A.Rotate(limit=12, border_mode=BC, p=1.0), CD(8, 24, 24)]),
        'rot_elastic':        A.Compose([_R, A.Rotate(limit=12, border_mode=BC, p=1.0), A.ElasticTransform(alpha=60, sigma=6, p=1.0)]),
        'rot_affine':         A.Compose([_R, A.Rotate(limit=12, border_mode=BC, p=1.0), A.Affine(scale=(0.85,1.15), translate_percent=(-0.1,0.1), shear=(-10,10), p=1.0)]),
        'rot_optical':        A.Compose([_R, A.Rotate(limit=12, border_mode=BC, p=1.0), A.OpticalDistortion(distort_limit=0.3, p=1.0)]),
        
        'affine_bright':      A.Compose([_R, A.Affine(scale=(0.85,1.15), translate_percent=(-0.1,0.1), shear=(-10,10), p=1.0), A.RandomBrightnessContrast(0.2, 0.2, p=1.0)]),
        'affine_noise':       A.Compose([_R, A.Affine(scale=(0.85,1.15), translate_percent=(-0.1,0.1), shear=(-10,10), p=1.0), GN(20, 80)]),
        'affine_blur':        A.Compose([_R, A.Affine(scale=(0.85,1.15), translate_percent=(-0.1,0.1), shear=(-10,10), p=1.0), A.GaussianBlur(blur_limit=(3,7), p=1.0)]),
        'affine_clahe':       A.Compose([_R, A.Affine(scale=(0.85,1.15), translate_percent=(-0.1,0.1), shear=(-10,10), p=1.0), A.CLAHE(clip_limit=4.0, p=1.0)]),
        'affine_gamma':       A.Compose([_R, A.Affine(scale=(0.85,1.15), translate_percent=(-0.1,0.1), shear=(-10,10), p=1.0), A.RandomGamma(gamma_limit=(80,140), p=1.0)]),
        'affine_sharpen':     A.Compose([_R, A.Affine(scale=(0.85,1.15), translate_percent=(-0.1,0.1), shear=(-10,10), p=1.0), A.Sharpen(alpha=(0.3,0.6), lightness=(0.8,1.0), p=1.0)]),
        'affine_equalize':    A.Compose([_R, A.Affine(scale=(0.85,1.15), translate_percent=(-0.1,0.1), shear=(-10,10), p=1.0), A.Equalize(p=1.0)]),
        'affine_solarize':    A.Compose([_R, A.Affine(scale=(0.85,1.15), translate_percent=(-0.1,0.1), shear=(-10,10), p=1.0), SOL(128)]),
        'affine_dropout':     A.Compose([_R, A.Affine(scale=(0.85,1.15), translate_percent=(-0.1,0.1), shear=(-10,10), p=1.0), CD(8, 24, 24)]),
        'affine_motion':      A.Compose([_R, A.Affine(scale=(0.85,1.15), translate_percent=(-0.1,0.1), shear=(-10,10), p=1.0), A.MotionBlur(blur_limit=9, p=1.0)]),
        
        'bright_clahe':       A.Compose([_R, A.RandomBrightnessContrast(0.2, 0.2, p=1.0), A.CLAHE(clip_limit=4.0, p=1.0)]),
        'bright_gamma':       A.Compose([_R, A.RandomBrightnessContrast(0.2, 0.2, p=1.0), A.RandomGamma(gamma_limit=(80,140), p=1.0)]),
        'bright_noise':       A.Compose([_R, A.RandomBrightnessContrast(0.2, 0.2, p=1.0), GN(20, 80)]),
        'bright_blur':        A.Compose([_R, A.RandomBrightnessContrast(0.2, 0.2, p=1.0), A.GaussianBlur(blur_limit=(3,7), p=1.0)]),
        'bright_sharpen':     A.Compose([_R, A.RandomBrightnessContrast(0.2, 0.2, p=1.0), A.Sharpen(alpha=(0.3,0.6), lightness=(0.8,1.0), p=1.0)]),
        'bright_equalize':    A.Compose([_R, A.RandomBrightnessContrast(0.2, 0.2, p=1.0), A.Equalize(p=1.0)]),
        'bright_motion':      A.Compose([_R, A.RandomBrightnessContrast(0.2, 0.2, p=1.0), A.MotionBlur(blur_limit=9, p=1.0)]),
        'clahe_noise':        A.Compose([_R, A.CLAHE(clip_limit=4.0, p=1.0), GN(20, 80)]),
        'clahe_blur':         A.Compose([_R, A.CLAHE(clip_limit=4.0, p=1.0), A.GaussianBlur(blur_limit=(3,7), p=1.0)]),
        'clahe_sharpen':      A.Compose([_R, A.CLAHE(clip_limit=4.0, p=1.0), A.Sharpen(alpha=(0.3,0.6), lightness=(0.8,1.0), p=1.0)]),
        'gamma_noise':        A.Compose([_R, A.RandomGamma(gamma_limit=(80,140), p=1.0), GN(20, 80)]),
        'gamma_blur':         A.Compose([_R, A.RandomGamma(gamma_limit=(80,140), p=1.0), A.GaussianBlur(blur_limit=(3,7), p=1.0)]),
        'gamma_sharpen':      A.Compose([_R, A.RandomGamma(gamma_limit=(80,140), p=1.0), A.Sharpen(alpha=(0.3,0.6), lightness=(0.8,1.0), p=1.0)]),
        'noise_blur':         A.Compose([_R, GN(20, 80), A.GaussianBlur(blur_limit=(3,7), p=1.0)]),
        'noise_motion':       A.Compose([_R, GN(20, 80), A.MotionBlur(blur_limit=9, p=1.0)]),
        'noise_sharpen':      A.Compose([_R, GN(20, 80), A.Sharpen(alpha=(0.3,0.6), lightness=(0.8,1.0), p=1.0)]),
        'blur_sharpen':       A.Compose([_R, A.GaussianBlur(blur_limit=(3,7), p=1.0), A.Sharpen(alpha=(0.3,0.6), lightness=(0.8,1.0), p=1.0)]),
        'dropout_noise':      A.Compose([_R, CD(8, 24, 24), GN(20, 80)]),
        'dropout_blur':       A.Compose([_R, CD(8, 24, 24), A.GaussianBlur(blur_limit=(3,7), p=1.0)]),
        
        
        'hflip_rot_bright':   A.Compose([_R, A.HorizontalFlip(p=1.0), A.Rotate(limit=12, border_mode=BC, p=1.0), A.RandomBrightnessContrast(0.2, 0.2, p=1.0)]),
        'hflip_rot_noise':    A.Compose([_R, A.HorizontalFlip(p=1.0), A.Rotate(limit=12, border_mode=BC, p=1.0), GN(20, 80)]),
        'hflip_rot_blur':     A.Compose([_R, A.HorizontalFlip(p=1.0), A.Rotate(limit=12, border_mode=BC, p=1.0), A.GaussianBlur(blur_limit=(3,7), p=1.0)]),
        'hflip_rot_clahe':    A.Compose([_R, A.HorizontalFlip(p=1.0), A.Rotate(limit=12, border_mode=BC, p=1.0), A.CLAHE(clip_limit=4.0, p=1.0)]),
        'hflip_rot_gamma':    A.Compose([_R, A.HorizontalFlip(p=1.0), A.Rotate(limit=12, border_mode=BC, p=1.0), A.RandomGamma(gamma_limit=(80,140), p=1.0)]),
        'hflip_rot_sharpen':  A.Compose([_R, A.HorizontalFlip(p=1.0), A.Rotate(limit=12, border_mode=BC, p=1.0), A.Sharpen(alpha=(0.3,0.6), lightness=(0.8,1.0), p=1.0)]),
        'hflip_rot_motion':   A.Compose([_R, A.HorizontalFlip(p=1.0), A.Rotate(limit=12, border_mode=BC, p=1.0), A.MotionBlur(blur_limit=9, p=1.0)]),
        'hflip_rot_equalize': A.Compose([_R, A.HorizontalFlip(p=1.0), A.Rotate(limit=12, border_mode=BC, p=1.0), A.Equalize(p=1.0)]),
        'hflip_rot_solarize': A.Compose([_R, A.HorizontalFlip(p=1.0), A.Rotate(limit=12, border_mode=BC, p=1.0), SOL(128)]),
        'hflip_rot_dropout':  A.Compose([_R, A.HorizontalFlip(p=1.0), A.Rotate(limit=12, border_mode=BC, p=1.0), CD(8, 24, 24)]),
        'rot_elastic_bright': A.Compose([_R, A.Rotate(limit=12, border_mode=BC, p=1.0), A.ElasticTransform(alpha=60, sigma=6, p=1.0), A.RandomBrightnessContrast(0.2, 0.2, p=1.0)]),
        'rot_elastic_noise':  A.Compose([_R, A.Rotate(limit=12, border_mode=BC, p=1.0), A.ElasticTransform(alpha=60, sigma=6, p=1.0), GN(20, 80)]),
        'rot_elastic_clahe':  A.Compose([_R, A.Rotate(limit=12, border_mode=BC, p=1.0), A.ElasticTransform(alpha=60, sigma=6, p=1.0), A.CLAHE(clip_limit=4.0, p=1.0)]),
        'rot_elastic_gamma':  A.Compose([_R, A.Rotate(limit=12, border_mode=BC, p=1.0), A.ElasticTransform(alpha=60, sigma=6, p=1.0), A.RandomGamma(gamma_limit=(80,140), p=1.0)]),
        'rot_affine_bright':  A.Compose([_R, A.Rotate(limit=12, border_mode=BC, p=1.0), A.Affine(scale=(0.85,1.15), translate_percent=(-0.1,0.1), shear=(-10,10), p=1.0), A.RandomBrightnessContrast(0.2, 0.2, p=1.0)]),
        'rot_affine_noise':   A.Compose([_R, A.Rotate(limit=12, border_mode=BC, p=1.0), A.Affine(scale=(0.85,1.15), translate_percent=(-0.1,0.1), shear=(-10,10), p=1.0), GN(20, 80)]),
        'rot_affine_clahe':   A.Compose([_R, A.Rotate(limit=12, border_mode=BC, p=1.0), A.Affine(scale=(0.85,1.15), translate_percent=(-0.1,0.1), shear=(-10,10), p=1.0), A.CLAHE(clip_limit=4.0, p=1.0)]),
        'affine_bright_noise':A.Compose([_R, A.Affine(scale=(0.85,1.15), translate_percent=(-0.1,0.1), shear=(-10,10), p=1.0), A.RandomBrightnessContrast(0.2, 0.2, p=1.0), GN(20, 80)]),
        'affine_bright_blur': A.Compose([_R, A.Affine(scale=(0.85,1.15), translate_percent=(-0.1,0.1), shear=(-10,10), p=1.0), A.RandomBrightnessContrast(0.2, 0.2, p=1.0), A.GaussianBlur(blur_limit=(3,7), p=1.0)]),
        'affine_bright_clahe':A.Compose([_R, A.Affine(scale=(0.85,1.15), translate_percent=(-0.1,0.1), shear=(-10,10), p=1.0), A.RandomBrightnessContrast(0.2, 0.2, p=1.0), A.CLAHE(clip_limit=4.0, p=1.0)]),
        'bright_noise_blur':  A.Compose([_R, A.RandomBrightnessContrast(0.2, 0.2, p=1.0), GN(20, 80), A.GaussianBlur(blur_limit=(3,7), p=1.0)]),
        'bright_clahe_noise': A.Compose([_R, A.RandomBrightnessContrast(0.2, 0.2, p=1.0), A.CLAHE(clip_limit=4.0, p=1.0), GN(20, 80)]),
        'bright_gamma_noise': A.Compose([_R, A.RandomBrightnessContrast(0.2, 0.2, p=1.0), A.RandomGamma(gamma_limit=(80,140), p=1.0), GN(20, 80)]),
        'clahe_gamma_noise':  A.Compose([_R, A.CLAHE(clip_limit=4.0, p=1.0), A.RandomGamma(gamma_limit=(80,140), p=1.0), GN(20, 80)]),
        'hflip_bright_noise': A.Compose([_R, A.HorizontalFlip(p=1.0), A.RandomBrightnessContrast(0.2, 0.2, p=1.0), GN(20, 80)]),
        'hflip_bright_clahe': A.Compose([_R, A.HorizontalFlip(p=1.0), A.RandomBrightnessContrast(0.2, 0.2, p=1.0), A.CLAHE(clip_limit=4.0, p=1.0)]),
        'hflip_bright_gamma': A.Compose([_R, A.HorizontalFlip(p=1.0), A.RandomBrightnessContrast(0.2, 0.2, p=1.0), A.RandomGamma(gamma_limit=(80,140), p=1.0)]),
        'hflip_clahe_noise':  A.Compose([_R, A.HorizontalFlip(p=1.0), A.CLAHE(clip_limit=4.0, p=1.0), GN(20, 80)]),
        'hflip_gamma_noise':  A.Compose([_R, A.HorizontalFlip(p=1.0), A.RandomGamma(gamma_limit=(80,140), p=1.0), GN(20, 80)]),
        'hflip_noise_blur':   A.Compose([_R, A.HorizontalFlip(p=1.0), GN(20, 80), A.GaussianBlur(blur_limit=(3,7), p=1.0)]),
        'hflip_dropout_noise':A.Compose([_R, A.HorizontalFlip(p=1.0), CD(8, 24, 24), GN(20, 80)]),
        'hflip_motion_bright':A.Compose([_R, A.HorizontalFlip(p=1.0), A.MotionBlur(blur_limit=9, p=1.0), A.RandomBrightnessContrast(0.2, 0.2, p=1.0)]),
        'rot30_bright_noise': A.Compose([_R, A.Rotate(limit=(30,30), border_mode=BC, p=1.0), A.RandomBrightnessContrast(0.2, 0.2, p=1.0), GN(20, 80)]),
        'rot30_clahe_noise':  A.Compose([_R, A.Rotate(limit=(30,30), border_mode=BC, p=1.0), A.CLAHE(clip_limit=4.0, p=1.0), GN(20, 80)]),
        'optical_bright_noise':A.Compose([_R, A.OpticalDistortion(distort_limit=0.3, p=1.0), A.RandomBrightnessContrast(0.2, 0.2, p=1.0), GN(20, 80)]),
        'optical_clahe_blur': A.Compose([_R, A.OpticalDistortion(distort_limit=0.3, p=1.0), A.CLAHE(clip_limit=4.0, p=1.0), A.GaussianBlur(blur_limit=(3,7), p=1.0)]),
        'grid_bright_noise':  A.Compose([_R, A.GridDistortion(num_steps=5, distort_limit=0.3, p=1.0), A.RandomBrightnessContrast(0.2, 0.2, p=1.0), GN(20, 80)]),
        'grid_clahe_noise':   A.Compose([_R, A.GridDistortion(num_steps=5, distort_limit=0.3, p=1.0), A.CLAHE(clip_limit=4.0, p=1.0), GN(20, 80)]),
    }


import warnings
with warnings.catch_warnings():
    warnings.simplefilter('error')
    try:
        test_p = build_pipelines(IMG_SIZE)
        print(f'Total pipelines available : {len(test_p)}')
        print(f'Pipelines that will be used: {VARIANTS_PER_PAIR}')
        assert len(test_p) >= VARIANTS_PER_PAIR
        print(f'Pipelines built cleanly (0 warnings)')
        del test_p
    except UserWarning as e:
        print(f'WARNING: {e}')

In [ ]:
def img_to_bytes(arr: np.ndarray) -> bytes:
    buf = io.BytesIO()
    Image.fromarray(arr).save(buf, format='PNG', compress_level=1)
    return buf.getvalue()


def augment_task(task_name, pairs, mask_subfolder):
    pipelines     = build_pipelines(IMG_SIZE)
    pipeline_keys = list(pipelines.keys())[:VARIANTS_PER_PAIR]

    all_jobs = [
        (img_p, msk_p, split, cat, suffix)
        for img_p, msk_p, split, cat in pairs
        for suffix in pipeline_keys
    ]

    n_chunks    = math.ceil(len(all_jobs) / PAIRS_PER_CHUNK)
    manifest    = []
    grand_total = 0
    t_start     = time.time()

    print(f'\n[{task_name}] {len(pairs):,} source pairs '
          f'× {len(pipeline_keys)} pipelines '
          f'= {len(all_jobs):,} total jobs')
    print(f'[{task_name}] Writing {n_chunks} zip chunk(s)...')

    for chunk_idx in range(n_chunks):
        chunk_jobs = all_jobs[chunk_idx * PAIRS_PER_CHUNK : (chunk_idx+1) * PAIRS_PER_CHUNK]
        zip_name   = f'{task_name}_chunk{chunk_idx+1:02d}_of_{n_chunks:02d}.zip'
        zip_path   = os.path.join(ZIP_DIR, zip_name)
        t0         = time.time()
        img_cache  = {}
        mask_cache = {}

        with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_STORED) as zf:
            pipe_objs = build_pipelines(IMG_SIZE)

            for i, (img_path, mask_path, split, cat, suffix) in enumerate(chunk_jobs):
                if img_path not in img_cache:
                    img_cache[img_path]   = enhance_xray(
                        np.array(Image.open(img_path).convert('RGB')))
                    mask_cache[mask_path] = np.array(
                        Image.open(mask_path).convert('L'))

                out  = pipe_objs[suffix](image=img_cache[img_path],
                                         mask=mask_cache[mask_path])
                base = os.path.splitext(os.path.basename(img_path))[0]

                zf.writestr(
                    f'{task_name}/{split}/{cat}/images/{base}_{suffix}.png',
                    img_to_bytes(out['image']))
                zf.writestr(
                    f'{task_name}/{split}/{cat}/{mask_subfolder}/{base}_{suffix}_mask.png',
                    img_to_bytes(out['mask']))

                grand_total += 1

                if (i+1) % 500 == 0 or (i+1) == len(chunk_jobs):
                    elapsed = time.time() - t0
                    rate    = (i+1) / max(elapsed, 1)
                    eta     = (len(chunk_jobs) - i - 1) / max(rate, 1)
                    zip_mb  = os.path.getsize(zip_path) / 1e6
                    free_gb = shutil.disk_usage('/kaggle/working').free / 1e9
                    print(f'  [{task_name}] chunk {chunk_idx+1}/{n_chunks} | '
                          f'{i+1:>5}/{len(chunk_jobs)} | '
                          f'{rate:.0f} p/s | '
                          f'ETA {eta/60:.1f}m | '
                          f'zip {zip_mb:.0f}MB | '
                          f'free {free_gb:.1f}GB')

        img_cache.clear()
        mask_cache.clear()

        final_mb = os.path.getsize(zip_path) / 1e6
        print(f'  [{task_name}] chunk {chunk_idx+1} done — '
              f'{final_mb:.0f} MB | {(time.time()-t0)/60:.1f} min')
        manifest.append({'file': zip_name,
                          'pairs': len(chunk_jobs),
                          'size_mb': round(final_mb)})

    print(f'[{task_name}] Complete — {grand_total:,} pairs in '
          f'{(time.time()-t_start)/60:.1f} min')
    return manifest


lung_manifest   = augment_task('lung',      lung_pairs,   'masks')
infect_manifest = augment_task('infection', infect_pairs, 'masks')

print(f'\nAll augmentation complete.')
print(f'Free space : {shutil.disk_usage("/kaggle/working").free / 1e9:.1f} GB')

In [ ]:
all_zips       = sorted(glob.glob(os.path.join(ZIP_DIR, '*.zip')))
total_zip_size = 0

print('═' * 62)
print('  ZIP SUMMARY')
print('═' * 62)
for zp in all_zips:
    with zipfile.ZipFile(zp, 'r') as zf:
        n_images = sum(1 for f in zf.namelist() if '/images/' in f)
    size_mb = os.path.getsize(zp) / 1e6
    total_zip_size += size_mb
    print(f'  {os.path.basename(zp):<46}  {n_images:>6,} pairs  {size_mb:>6.0f} MB')
print(f'  {"─"*60}')
print(f'  Total : {total_zip_size:.0f} MB across {len(all_zips)} zip(s)')
print(f'  Free  : {shutil.disk_usage("/kaggle/working").free / 1e9:.1f} GB remaining')
print('═' * 62)

manifest = {
    'lung_chunks':             lung_manifest,
    'infection_chunks':        infect_manifest,
    'img_size':                IMG_SIZE,
    'variants_per_pair':       VARIANTS_PER_PAIR,
    'splits':                  SPLITS,
    'categories':              CATEGORIES,
    'lung_source_pairs':       len(lung_pairs),
    'infection_source_pairs':  len(infect_pairs),
    'lung_total_images':       len(lung_pairs) * VARIANTS_PER_PAIR,
    'infection_total_images':  len(infect_pairs) * VARIANTS_PER_PAIR,
}
manifest_path = os.path.join(ZIP_DIR, 'manifest.json')
with open(manifest_path, 'w') as f:
    json.dump(manifest, f, indent=2)
print(f'\nManifest saved : {manifest_path}')

In [ ]:
import shutil, os, json, subprocess
from kaggle_secrets import UserSecretsClient

secrets         = UserSecretsClient()
kaggle_username = secrets.get_secret('KAGGLE_USERNAME')
kaggle_key      = secrets.get_secret('KAGGLE_KEY')

os.makedirs('/root/.config/kaggle', exist_ok=True)
with open('/root/.config/kaggle/kaggle.json', 'w') as f:
    json.dump({'username': kaggle_username, 'key': kaggle_key}, f)
os.chmod('/root/.config/kaggle/kaggle.json', 0o600)

# Write metadata to /tmp (outside the full disk)
metadata = {
    'title':     'COVID-QU-Ex Augmented Segmentation',
    'id':        f'{kaggle_username}/covidqu-augmented',
    'licenses':  [{'name': 'CC0-1.0'}],
    'isPrivate': True,
}
with open('/tmp/dataset-metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

# Remove existing symlink/file if it exists, then recreate
symlink_path = '/kaggle/working/zips/dataset-metadata.json'
if os.path.exists(symlink_path) or os.path.islink(symlink_path):
    os.remove(symlink_path)

os.symlink('/tmp/dataset-metadata.json', symlink_path)
print(f'Free space : {shutil.disk_usage("/kaggle/working").free / 1e9:.1f} GB')

print('Publishing...')
result = subprocess.run(
    ['kaggle', 'datasets', 'create',
     '-p', '/kaggle/working/zips', '--dir-mode', 'zip'],
    capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)
else:
    print(f'Published!')
    print(f'URL: https://www.kaggle.com/datasets/{kaggle_username}/covidqu-augmented')

In [ ]:
result = subprocess.run(
    ['kaggle', 'datasets', 'version',
     '-p', DATASET_DIR,
     '-m', 'Updated augmented COVID-QU-Ex dataset',
     '--dir-mode', 'zip'],
    capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)
else:
    print('Version updated successfully.')